# falsefaced: activation extraction (PLAN.md step 3-4)

Runs `scripts/extract.py` from the public repo on 2x T4: Neural Chameleon checkpoint in fp16 with eager attention, three concepts x three trigger conditions, 64 generated tokens, mean layer-12 residual. Outputs land in `/kaggle/working/data/` (`acts.npy`, `meta.csv`, `extract_config.json`) and are fetched with `kaggle kernels output`. Set `LIMIT = 10` for the fp16 test run, `LIMIT = None` for all prompts.

In [ ]:
LIMIT = None      # prompts per concept; None = all (~500 per concept), file order
BATCH = 8
REPO = "https://github.com/star2vec/falsefaced"

In [ ]:
import os
os.environ["HF_HOME"] = "/kaggle/tmp/hf"            # keep the 18.5 GB checkpoint out of the output dir
try:                                                  # optional: only the gated fallback base model needs it
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN secret: attached")
except Exception as e:
    print("HF_TOKEN secret: not attached (fine, the checkpoint is public):", type(e).__name__)
!rm -rf /kaggle/tmp/falsefaced && git clone -q {REPO} /kaggle/tmp/falsefaced && git -C /kaggle/tmp/falsefaced log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -c "import torch, transformers; print('torch', torch.__version__, '| transformers', transformers.__version__, '| gpus', torch.cuda.device_count())"

In [ ]:
%cd /kaggle/tmp/falsefaced
limit = f"--limit {LIMIT}" if LIMIT else ""
!python scripts/extract.py {limit} --batch-size {BATCH} --dtype fp16 --out /kaggle/working/data

In [ ]:
!ls -la /kaggle/working/data && cat /kaggle/working/data/extract_config.json